# Feature Engineering & Data Preparation
**Digital Banking Fraud Detection Project**

This notebook prepares the dataset for model training by dropping identifiers, engineering time features, performing one-hot encoding, saving configuration files for deployment, scaling features, and addressing class imbalance via SMOTE.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

### Load the Dataset

In [ ]:
df = pd.read_csv('../Dataset/Bank_Transaction_Fraud_Detection.csv')
print("Shape:", df.shape)

### Drop Identifiers, PII & High Granularity Features

In [ ]:
pii_cols = ['Customer_ID', 'Customer_Name', 'Customer_Contact', 'Customer_Email',
            'Transaction_ID', 'Merchant_ID', 'Transaction_Description',
            'Bank_Branch', 'Transaction_Location']
df.drop(columns=pii_cols, inplace=True)
print("Remaining columns:", df.shape[1])

### Date/Time Feature Extraction

In [ ]:
df['Parsed_Date'] = pd.to_datetime(df['Transaction_Date'], format='%d-%m-%Y', errors='coerce')
df['Transaction_Hour'] = df['Transaction_Time'].str.split(':').str[0].astype(int)
df['Transaction_DayOfWeek'] = df['Parsed_Date'].dt.dayofweek
df['Transaction_Month'] = df['Parsed_Date'].dt.month
df['Transaction_DayOfMonth'] = df['Parsed_Date'].dt.day

df.drop(columns=['Transaction_Date', 'Transaction_Time', 'Parsed_Date'], inplace=True)
df.head()

### Save Categorical Unique Values (For Streamlit UI Mapping)

In [ ]:
categorical_cols = ['Gender', 'State', 'City', 'Account_Type', 'Transaction_Type', 
                    'Merchant_Category', 'Transaction_Device', 'Device_Type', 'Transaction_Currency']

categorical_options = {col: sorted(list(df[col].dropna().unique())) for col in categorical_cols}

os.makedirs('../Streamlit-app', exist_ok=True)
with open('../Streamlit-app/categorical_options.pkl', 'wb') as f:
    pickle.dump(categorical_options, f)
print("Saved categorical_options.pkl for Streamlit UI.")

### One-Hot Encoding

In [ ]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False)
print("Shape after encoding:", df_encoded.shape)
df_encoded.head()

### Train/Test Split (Stratified)

In [ ]:
X = df_encoded.drop(columns=['Is_Fraud'])
y = df_encoded['Is_Fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

### Scale Numeric Features

In [ ]:
numeric_cols = ['Age', 'Transaction_Amount', 'Account_Balance', 'Transaction_Hour', 
                'Transaction_DayOfWeek', 'Transaction_Month', 'Transaction_DayOfMonth']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Save metadata and scaler
with open('../Streamlit-app/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('../Streamlit-app/feature_columns.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)
with open('../Streamlit-app/numeric_columns.pkl', 'wb') as f:
    pickle.dump(numeric_cols, f)
print("Scaler and features saved.")

### Class Imbalance Handling (SMOTE on Train Only)

In [ ]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("Original training target distribution:")
print(y_train.value_counts())
print("\nResampled training target distribution:")
print(y_train_resampled.value_counts())

### Save Processed Data for Model Training

In [ ]:
os.makedirs('../Models', exist_ok=True)
X_train_resampled.to_csv('../Models/X_train.csv', index=False)
y_train_resampled.to_csv('../Models/y_train.csv', index=False)
X_test_scaled.to_csv('../Models/X_test.csv', index=False)
y_test.to_csv('../Models/y_test.csv', index=False)
print("Preprocessing finished. Data saved.")